In [1]:
from mlflow.tracking import MlflowClient


MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

### Interacting with the MLflow tracking server

The MlflowClient object allows us to interact with...

+ an MLflow Tracking Server that creates and manages experiments and runs.
+ an MLflow Registry Server that creates and manages registered models and model versions.
To instantiate it we need to pass a tracking URI and/or a registry URI

In [4]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='/home/mlops/mlops-project/mlflow/experiment-tracking/mlruns/1', creation_time=1786404408718, experiment_id='1', last_update_time=1786404408718, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1786326364166, experiment_id='0', last_update_time=1786326364166, lifecycle_stage='active', name='Default', tags={}>]

In [5]:
client.create_experiment(name="new-experiment")

'2'

In [6]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 7",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.rmse ASC"]
)

In [12]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: e2cc15a703434583b5fc4a8db54027ed, rmse: 5.1128
run id: 5b2e68994e0c414ea345c0e85a3fc165, rmse: 5.1273
run id: b67ece0b89b44a4d8f458dd1a7c6b26f, rmse: 5.1288
run id: 67dfa582b6f34ff889ddf3a29eb1b0cf, rmse: 5.1292
run id: b2a637c1fcf1424ea44e11ac84a80c35, rmse: 5.1300


### Interacting with the Model Registry

In this section We will use the MlflowClient instance to:
+ Register a new model for the experiment nyc-taxi-regressor
+ Retrieve the latests versions of the model nyc-taxi-regressor and check that new versions were created.
+ Set `champion` and `challenger` versions of the model

In [9]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [101]:
run_id = "d73514a8bea44b7e8f375e61849db796"
experiment_id = "1"

client.search_logged_models([experiment_id])

model_list = client.search_logged_models(
    experiment_ids=[experiment_id],
    filter_string=f"source_run_id = '{run_id}'",
    max_results=1000,
)

for model in model_list:
    print(model)

LoggedModel(artifact_location='/home/mlops/mlops-project/mlflow/experiment-tracking/mlruns/1/models/m-42c1b8dfe263496daee0b2355a2cea81/artifacts', creation_timestamp=1786424525987, experiment_id='1', last_updated_timestamp=1786424535581, model_id='m-42c1b8dfe263496daee0b2355a2cea81', model_type=None, model_uri='models:/m-42c1b8dfe263496daee0b2355a2cea81', name='model', source_run_id='d73514a8bea44b7e8f375e61849db796', status=<LoggedModelStatus.READY: 'READY'>, status_message=None)


In [ ]:
registered_model_name="nyc-taxi-regressor"
client.create_registered_model(name=registered_model_name)

In [103]:
selected_model_id = "m-42c1b8dfe263496daee0b2355a2cea81"
selected_model = client.get_logged_model(selected_model_id)
    
champion_model = client.create_model_version(
    name=registered_model_name,
    source=selected_model.model_uri,
    run_id=selected_model.source_run_id
)
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=champion_model.version,
)

challenger_model = client.create_model_version(
    name=registered_model_name,
    source=selected_model.model_uri,
    run_id=selected_model.source_run_id
)
client.set_registered_model_alias(
    name=model_name,
    alias="challenger",
    version=challenger_model.version,
)

In [104]:
def print_model_info(name):

    models = client.search_model_versions(
        filter_string=f"name = '{name}'",
        order_by=["version_number ASC"],
    )
    
    print(f"Model name: '{name}'")
    
    for model in models:
        version_info = client.get_model_version(
            name=model.name,
            version=model.version,
        )
    
        print(
            f"version={version_info.version}, "
            f"aliases={version_info.aliases}, "
            f"source={version_info.source}"
        )

In [105]:
print_model_info(registered_model_name)

Model name: 'nyc-taxi-regressor'
version=1, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=2, aliases=[], source=models:/nyc-taxi-regressor/1
version=3, aliases=[], source=models:/nyc-taxi-regressor/2
version=4, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=5, aliases=['champion'], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=6, aliases=['challenger'], source=models:/m-42c1b8dfe263496daee0b2355a2cea81


### Comparing versions and selecting the new champion model

The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:
+ Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2023.
+ Download the DictVectorizer that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
+ Preprocess the test set using the DictVectorizer so we can properly feed the regressors.
+ Make predictions on the test set using the model versions that are currently labeled as `challnger` and `champion`, and compare their performance.
+ Based on the results, set the new `champion` model version accordingly.

In [106]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import pickle


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(version_info, df, dv):
    client.download_artifacts(run_id=version_info.run_id, path='preprocessor', dst_path='.')

    with open("preprocessor/preprocessor.b", "rb") as f_in:
        dv = pickle.load(f_in)

    X_test = preprocess(df, dv)

    target = "duration"
    y_test = df[target].values

    model = mlflow.pyfunc.load_model(version_info.source)
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [107]:
df = read_dataframe("data/green_tripdata_2023-03.parquet")

In [108]:
challenger_version_info = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="challenger",
)

champion_version_info = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="champion",
)

In [109]:
%time test_model(version_info=challenger_version_info, df=df, dv=dv)

CPU times: user 7.36 s, sys: 2.77 s, total: 10.1 s
Wall time: 10.1 s


{'rmse': 5.716225073043043}

In [110]:
%time test_model(version_info=champion_version_info, df=df, dv=dv)

CPU times: user 7.09 s, sys: 703 ms, total: 7.79 s
Wall time: 7.78 s


{'rmse': 5.716225073043043}

In [111]:
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=challenger_version_info.version,
)

client.delete_registered_model_alias(
    name=model_name,
    alias="challenger",
)

print_model_info(registered_model_name)

Model name: 'nyc-taxi-regressor'
version=1, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=2, aliases=[], source=models:/nyc-taxi-regressor/1
version=3, aliases=[], source=models:/nyc-taxi-regressor/2
version=4, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=5, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=6, aliases=['champion'], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
